# 🎙️ BookVoice-AI — XTTS-v2 Fine-tuning

**Dieses Notebook trainiert eine eigene Stimme auf Basis deines BookVoice-Trainingsraum-Datasets.**

### Voraussetzungen
- ✅ Google Colab mit **T4 GPU** (Laufzeit → Laufzeittyp → T4)
- ✅ BookVoice-AI Server läuft und ist erreichbar
- ✅ Dataset wurde im Trainingsraum erstellt und exportiert

### Ablauf
1. Server & Projekt konfigurieren
2. Verbindung testen
3. Dataset herunterladen
4. Umgebung installieren (~5-10 Min)
5. Training starten
6. Modell exportieren & hochladen

In [ ]:
# ============================================================
# ZELLE 1: Server & Projekt konfigurieren
# ============================================================

# 🔧 Hier deine Einstellungen eintragen:
SERVER_URL = "https://ahrar.aksoy-net.de"  # z.B. https://deine-domain.de oder http://IP:7502
PROJEKT_NAME = "sufi_rumi"                  # Projektname aus dem Trainingsraum
TRAINING_PASSWORD = "sufi2026"              # Trainingsraum-Passwort (Standard: sufi2026)
MODELL_NAME = PROJEKT_NAME + "_finetuned"   # Name für das fertige Modell

# Training-Parameter
BATCH_SIZE = 4
EPOCHS = 50
LEARNING_RATE = "5e-6"

print(f"✅ Konfiguration:")
print(f"   Server:   {SERVER_URL}")
print(f"   Projekt:  {PROJEKT_NAME}")
print(f"   Modell:   {MODELL_NAME}")
print(f"   Batch:    {BATCH_SIZE} | Epochs: {EPOCHS} | LR: {LEARNING_RATE}")

In [ ]:
# ============================================================
# ZELLE 2: Verbindung testen
# ============================================================
import requests
import json

print("🔍 Teste Verbindung zum BookVoice-Server...")

# Health-Check
try:
    r = requests.get(f"{SERVER_URL}/health", timeout=10)
    if r.status_code == 200:
        print(f"✅ Server erreichbar: {SERVER_URL}")
    else:
        print(f"⚠️ Server antwortet mit Status {r.status_code}")
except Exception as e:
    print(f"❌ Server nicht erreichbar: {e}")
    print("   Prüfe ob der Server läuft und die URL korrekt ist.")
    raise

# Trainingsraum-Auth
print("\n🔐 Authentifizierung...")
r = requests.post(
    f"{SERVER_URL}/training/auth",
    json={"password": TRAINING_PASSWORD}
)
if r.status_code == 200:
    print("✅ Trainingsraum-Zugang OK")
else:
    print(f"❌ Falsches Passwort! Status: {r.status_code}")
    raise Exception("Authentifizierung fehlgeschlagen")

# Projekt prüfen
print(f"\n📁 Prüfe Projekt '{PROJEKT_NAME}'...")
r = requests.get(f"{SERVER_URL}/training/projects/{PROJEKT_NAME}")
if r.status_code == 200:
    p = r.json()
    print(f"✅ Projekt gefunden:")
    print(f"   Sprache:      {p.get('sprache', '?')}")
    print(f"   Clips:        {p.get('clips', 0)}")
    print(f"   Dataset:      {'✅ bereit' if p.get('dataset_bereit') else '❌ noch nicht exportiert'}")
    if not p.get('dataset_bereit'):
        print("\n⚠️ Dataset noch nicht exportiert!")
        print("   Bitte zuerst im Trainingsraum 'Dataset bauen' klicken.")
        raise Exception("Dataset nicht exportiert")
else:
    print(f"❌ Projekt nicht gefunden: {PROJEKT_NAME}")
    raise Exception("Projekt nicht gefunden")

In [ ]:
# ============================================================
# ZELLE 3: Dataset herunterladen & entpacken
# ============================================================
import os
import zipfile
import shutil

DATASET_DIR = "/content/dataset"
ZIP_PATH = "/content/dataset.zip"

# Alten Dataset-Ordner löschen falls vorhanden
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
    print("🗑️ Alter Dataset-Ordner gelöscht")

print(f"📥 Lade Dataset herunter...")
r = requests.get(
    f"{SERVER_URL}/training/projects/{PROJEKT_NAME}/export/download",
    stream=True
)
if r.status_code != 200:
    raise Exception(f"Download fehlgeschlagen: {r.status_code}")

total = int(r.headers.get('content-length', 0))
downloaded = 0
with open(ZIP_PATH, 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)
        downloaded += len(chunk)
        if total > 0:
            pct = downloaded / total * 100
            print(f"\r   {pct:.1f}% ({downloaded/1024:.0f} KB)", end="")

print(f"\n✅ Download abgeschlossen: {os.path.getsize(ZIP_PATH)/1024:.0f} KB")

# Entpacken
print("📦 Entpacke Dataset...")
os.makedirs(DATASET_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DATASET_DIR)

# Sprache + Speaker aus Dataset lesen
lang_file = os.path.join(DATASET_DIR, "lang.txt")
SPRACHE = open(lang_file).read().strip() if os.path.exists(lang_file) else "tr"

train_csv = os.path.join(DATASET_DIR, "metadata_train.csv")
SPEAKER_NAME = "sufi"
if os.path.exists(train_csv):
    with open(train_csv) as f:
        lines = f.readlines()
        if len(lines) > 1:
            SPEAKER_NAME = lines[1].strip().split('|')[-1]

# Statistik
wavs = [f for f in os.listdir(os.path.join(DATASET_DIR, "wavs")) if f.endswith(".wav")] if os.path.exists(os.path.join(DATASET_DIR, "wavs")) else []
train_lines = len(open(train_csv).readlines()) - 1 if os.path.exists(train_csv) else 0
eval_csv = os.path.join(DATASET_DIR, "metadata_eval.csv")
eval_lines = len(open(eval_csv).readlines()) - 1 if os.path.exists(eval_csv) else 0

print(f"\n📊 Dataset-Übersicht:")
print(f"   Sprache:     {SPRACHE}")
print(f"   Speaker:     {SPEAKER_NAME}")
print(f"   WAV-Clips:   {len(wavs)}")
print(f"   Train-Paare: {train_lines}")
print(f"   Eval-Paare:  {eval_lines}")
print(f"   Pfad:        {DATASET_DIR}")

In [ ]:
# ============================================================
# ZELLE 4: Umgebung installieren (~5-10 Minuten)
# ============================================================
print("⏳ Installiere XTTS Fine-tuning Umgebung...")
print("   Das dauert 5-10 Minuten — bitte warten!")
print()

In [ ]:
%%capture install_output
# XTTS Fine-tuning Repo klonen
!git clone https://github.com/daswer123/xtts-finetune-webui.git /content/xtts-finetune-webui

# Abhängigkeiten installieren
%cd /content/xtts-finetune-webui
!pip install -r requirements.txt
!pip install gradio==3.50.2 --quiet

In [ ]:
# Installation prüfen
import subprocess
result = subprocess.run(["python", "-c", "import torch; print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')"], capture_output=True, text=True, cwd="/content/xtts-finetune-webui")
print(result.stdout)
if "True" in result.stdout:
    print("✅ GPU verfügbar — Training wird schnell sein!")
else:
    print("⚠️ Keine GPU! Bitte Laufzeit auf T4 umstellen.")

In [ ]:
# ============================================================
# ZELLE 5: Training starten
# ============================================================
import subprocess
import threading
import time

print("🚀 Starte XTTS Fine-tuning WebUI...")
print()
print(f"📋 Training-Parameter:")
print(f"   Dataset:      {DATASET_DIR}")
print(f"   Sprache:      {SPRACHE}")
print(f"   Speaker:      {SPEAKER_NAME}")
print(f"   Batch Size:   {BATCH_SIZE}")
print(f"   Epochs:       {EPOCHS}")
print(f"   Learning Rate:{LEARNING_RATE}")
print()
print("⏳ Starte Gradio UI — warte auf die URL...")

# Gradio im Hintergrund starten
process = subprocess.Popen(
    ["python", "app.py", "--share"],
    cwd="/content/xtts-finetune-webui",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Auf Gradio-URL warten
gradio_url = None
for _ in range(60):
    line = process.stdout.readline()
    if "gradio.live" in line or "Running on" in line:
        gradio_url = [l for l in line.split() if "http" in l]
        if gradio_url:
            gradio_url = gradio_url[0]
            break
    time.sleep(1)

if gradio_url:
    print(f"\n✅ Gradio UI läuft!")
    print(f"   🌐 URL: {gradio_url}")
    print()
    print("📝 Anleitung im Gradio UI:")
    print(f"   1. Tab 'Train' öffnen")
    print(f"   2. Dataset Path: {DATASET_DIR}")
    print(f"   3. Language: {SPRACHE}")
    print(f"   4. Speaker Name: {SPEAKER_NAME}")
    print(f"   5. Batch Size: {BATCH_SIZE}")
    print(f"   6. Epochs: {EPOCHS}")
    print(f"   7. Learning Rate: {LEARNING_RATE}")
    print(f"   8. 'Start Training' klicken")
    print()
    print("⏱️ Training dauert ca. 2-3 Stunden auf T4 GPU")
    print("   Wenn fertig → Zelle 6 ausführen")
else:
    print("⚠️ Gradio URL nicht gefunden — prüfe Ausgabe:")
    print(process.stdout.read(500))

In [ ]:
# ============================================================
# ZELLE 6: Modell exportieren & zu BookVoice hochladen
# ============================================================
# ⚠️ Erst ausführen wenn Training abgeschlossen ist!
import os
import glob
import zipfile
import requests

print("🔍 Suche trainiertes Modell...")

# Modell-Pfade suchen (daswer123 speichert in run/)
model_dirs = [
    "/content/xtts-finetune-webui/run/training/GPT_XTTS_FT/",
    "/content/xtts-finetune-webui/output/",
]

model_pth = None
config_json = None
vocab_json = None

for d in model_dirs:
    ptns = glob.glob(os.path.join(d, "**", "*.pth"), recursive=True)
    if ptns:
        model_pth = sorted(ptns)[-1]  # neueste
        base = os.path.dirname(model_pth)
        config_json = os.path.join(base, "config.json")
        vocab_json = glob.glob(os.path.join("/content/xtts-finetune-webui", "**", "vocab.json"), recursive=True)
        vocab_json = vocab_json[0] if vocab_json else None
        break

if not model_pth:
    print("❌ Kein trainiertes Modell gefunden!")
    print("   Stelle sicher dass das Training abgeschlossen ist.")
    raise Exception("Modell nicht gefunden")

print(f"✅ Modell gefunden:")
print(f"   model.pth:   {model_pth}")
print(f"   config.json: {config_json}")
print(f"   vocab.json:  {vocab_json}")

# ZIP erstellen
ZIP_OUT = f"/content/{MODELL_NAME}.zip"
print(f"\n📦 Erstelle ZIP: {ZIP_OUT}")

with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(model_pth, "model.pth")
    if config_json and os.path.exists(config_json):
        z.write(config_json, "config.json")
    if vocab_json and os.path.exists(vocab_json):
        z.write(vocab_json, "vocab.json")

zip_size = os.path.getsize(ZIP_OUT) / 1024 / 1024
print(f"✅ ZIP erstellt: {zip_size:.1f} MB")

# Zu BookVoice hochladen
print(f"\n📤 Lade Modell zu BookVoice hoch...")
print(f"   URL: {SERVER_URL}/training/models/upload")

with open(ZIP_OUT, 'rb') as f:
    r = requests.post(
        f"{SERVER_URL}/training/models/upload",
        data={"name": MODELL_NAME},
        files={"file": (f"{MODELL_NAME}.zip", f, "application/zip")},
        timeout=300
    )

if r.status_code == 200:
    d = r.json()
    print(f"\n✅ Modell erfolgreich hochgeladen!")
    print(f"   Name:     {d.get('name')}")
    print(f"   Dateien:  {', '.join(d.get('dateien', []))}")
    print()
    print(f"🎉 Fertig! Jetzt in BookVoice-AI:")
    print(f"   1. Trainingsraum öffnen")
    print(f"   2. Karte '4 · Trainiertes Modell'")
    print(f"   3. Modell '{d.get('name')}' → 'Aktivieren'")
    print(f"   4. Hörbuch generieren mit deiner eigenen Stimme! 🎙️")
else:
    print(f"❌ Upload fehlgeschlagen: {r.status_code}")
    print(r.text[:500])
    print()
    print("📥 Alternativ: ZIP manuell herunterladen:")
    from google.colab import files
    files.download(ZIP_OUT)

---
## 📝 Hinweise

### Falls der Upload fehlschlägt
Die ZIP-Datei wird automatisch zum Download angeboten. Du kannst sie dann manuell im Trainingsraum hochladen.

### Training-Tipps
- **Mindestens 30 Clip-Paare** für gute Ergebnisse
- **Ruhige Aufnahmen** ohne Hintergrundgeräusche
- **Gleiche Stimme** in allen Clips
- Bei schlechten Ergebnissen: Epochs erhöhen (100) oder Learning Rate reduzieren (1e-6)

### Modell aktivieren
Nach dem Upload in BookVoice-AI:
1. 🎓 Trainingsraum → **4 · Trainiertes Modell**
2. Modell in der Liste → **Aktivieren**
3. Studio → Hörbuch generieren

### Zurück zum Basis-Modell
Im Trainingsraum → **Basis-Modell aktivieren**